# Baseline Models — Random Forest & MaxEntThis notebook trains two traditional species distribution modelling baselines — **Random Forest** and **MaxEnt** — using the same cross-validation, model selection, ensemble, and mapping pipeline.## Pipeline (shared by both models)1. **K-fold CV** with hyperparameter stage search  2. **Per-fold model selection**: AUC floor → logloss window → max Boyce (tie-break: lower logloss)  3. **OOF metrics** (Boyce + AUC)  4. **AUC-weighted ensemble** on held-out test  5. **Tiled map prediction** from pre-standardised rasters  | Model | Hyperparameter searched | Prediction type ||-------|------------------------|-----------------|| Random Forest | `n_estimators` | `predict_proba(X)[:, 1]` || MaxEnt (elapid) | `beta_multiplier` | `model.predict(X)` (cloglog / logistic) |

## 1. Imports

In [ ]:
import os
import warnings
from pathlib import Path
from typing import Sequence, Dict, Tuple, Iterable, Optional, Union

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import log_loss

import rasterio
from rasterio.windows import Window
from tqdm import tqdm

import elapid as ela

# Must return dict with keys "Boyce" and "ROC_AUC"
from standard_metrics import compute_metrics

## 2. Configuration

In [ ]:
# ── Paths ─────────────────────────────────────────────────
TRAIN_PATH = "data/features/cv_point_features.parquet"
TEST_PATH  = "data/features/test_point_features.parquet"

PROCESSED_RASTER_DIR = "data/covariates/processed"
LANDMASK_TIF         = "data/covariates/landmask.tif"
OUT_DIR_RF           = "outputs/random_forest"
OUT_DIR_MAXENT       = "outputs/maxent"

# ── Column names ──────────────────────────────────────────
FOLD_COL  = "fold"
LABEL_COL = "mod_label"
LON_COL   = "x"
LAT_COL   = "y"
DROP_COLS = ("Longitude", "Latitude", "label", "cluster_id")

# ── Shared training settings ─────────────────────────────
SEED           = 42
NBINS_BOYCE    = 20
EPS_PROBA      = 1e-7

USE_AUC_FLOOR     = True
AUC_FLOOR         = 0.80
LOSS_WINDOW_DELTA = 0.03

# ── Ensemble ──────────────────────────────────────────────
AUC_WEIGHT_SHIFT_BY_MIN = True

# ── Mapping ───────────────────────────────────────────────
TILE           = 1024
DST_NODATA     = -9999.0
LANDMASK_VALUE = 1

# ── RF-specific ───────────────────────────────────────────
TREE_STAGES = (500, 1000, 1500, 2000, 2500, 3000, 3500, 4000, 4500, 5000)
RF_PARAMS = {
    "random_state": SEED,
    "n_jobs": -1,
    "bootstrap": True,
    "max_depth": None,
    "min_samples_leaf": 10,
    "min_samples_split": 20,
    "max_features": "sqrt",
}
CLASS_WEIGHT_BALANCED = False

# ── MaxEnt-specific ───────────────────────────────────────
BETA_STAGES      = (0.5, 1.0, 1.5, 2.0, 3.0, 4.0)
MAXENT_TRANSFORM = "cloglog"   # or "logistic"

np.random.seed(SEED)

## 3. Shared UtilitiesLabel coercion, feature selection, NaN handling, raster grid checks, and tiling — all used identically by both models.

In [ ]:
# ── Label coercion ────────────────────────────────────────

def ensure_binary_label(df: pd.DataFrame, col=LABEL_COL) -> None:
    """Coerce a label column to 0/1 integers in-place."""
    if df[col].dtype == object:
        mapper = {
            "presence": 1, "pres": 1, "pos": 1, "positive": 1,
            "1": 1, "true": 1, "t": 1, "yes": 1,
            "background": 0, "absence": 0, "abs": 0, "bg": 0,
            "neg": 0, "negative": 0, "0": 0, "false": 0, "f": 0, "no": 0,
        }
        df[col] = df[col].astype(str).str.lower().str.strip().map(mapper)
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int).clip(0, 1)


# ── Feature selection ────────────────────────────────────

def select_numeric_features(df, label_col=LABEL_COL, fold_col=FOLD_COL,
                            lon_col=LON_COL, lat_col=LAT_COL, drop_cols=DROP_COLS):
    """Select numeric columns with non-zero variance, excluding metadata."""
    exclude = {label_col, fold_col, lon_col, lat_col, *drop_cols}
    keep = []
    for c in df.columns:
        if c in exclude or not np.issubdtype(df[c].dtype, np.number):
            continue
        v = df[c].to_numpy()
        if not np.isfinite(v).any() or np.nanstd(v, ddof=0) == 0:
            continue
        keep.append(c)
    return keep


# ── NaN handling ─────────────────────────────────────────

def _nan_to_zero(X):
    """Replace NaN/inf with 0 in numpy array or DataFrame."""
    if isinstance(X, pd.DataFrame):
        return X.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)


# ── Raster grid helpers ──────────────────────────────────

def _grid_signature(src):
    return (str(src.crs), tuple(src.transform)[:6], src.width, src.height, src.nodata)

def _same_grid(sig_a, sig_b, atol=1e-9):
    crs_a, t_a, w_a, h_a, _ = sig_a
    crs_b, t_b, w_b, h_b, _ = sig_b
    if crs_a != crs_b or w_a != w_b or h_a != h_b:
        return False
    return np.allclose(np.array(t_a), np.array(t_b), atol=atol, rtol=0)

def tiles(height, width, tile):
    for r0 in range(0, height, tile):
        for c0 in range(0, width, tile):
            yield r0, min(r0 + tile, height), c0, min(c0 + tile, width)

## 4. Generic Stage-Search Model SelectionThe model selection logic is identical for both RF and MaxEnt — only the model fitting and prediction differ. We factor out the selection into a generic function that takes lists of (AUC, Boyce, logloss) per stage.

In [ ]:
def select_best_stage(
    val_auc: np.ndarray,
    val_boyce: np.ndarray,
    val_logloss: np.ndarray,
    use_auc_floor: bool = USE_AUC_FLOOR,
    auc_floor: float = AUC_FLOOR,
    loss_window_delta: float = LOSS_WINDOW_DELTA,
) -> int:
    """
    Select the best stage index using the shared selection rule:
      1. Keep stages with AUC >= floor (if enabled)
      2. Among those, find min logloss and keep within window
      3. Pick max Boyce; tie-break lower logloss
      4. Fallback: best AUC; fallback: index 0
    """
    finite_auc = np.isfinite(val_auc)
    finite_ll  = np.isfinite(val_logloss)

    ok = np.zeros_like(val_auc, dtype=bool)
    if use_auc_floor and finite_auc.any():
        ok = finite_auc & (val_auc >= auc_floor)
    ok = ok & finite_ll

    if ok.any():
        ll_min = float(np.min(val_logloss[ok]))
        in_window = ok & (val_logloss <= ll_min + loss_window_delta)
        idxs = np.where(in_window)[0]

        boy_eff = np.where(np.isfinite(val_boyce), val_boyce, -np.inf)
        best_idx, best_tuple = None, (-np.inf, np.inf)
        for i in idxs:
            tup = (float(boy_eff[i]), float(val_logloss[i]))
            if tup[0] > best_tuple[0] or (tup[0] == best_tuple[0] and tup[1] < best_tuple[1]):
                best_tuple, best_idx = tup, int(i)
        return best_idx
    elif finite_auc.any():
        return int(np.nanargmax(val_auc))
    else:
        return 0


def _eval_stage(y_va, p_va):
    """Compute AUC, Boyce, logloss for one stage's validation predictions."""
    if np.unique(y_va).size < 2 or not np.isfinite(p_va).all():
        return np.nan, np.nan, np.nan
    mets = compute_metrics(y_va, p_va, nbins_boyce=NBINS_BOYCE)
    auc = float(mets["ROC_AUC"])
    boy = float(mets["Boyce"])
    ll  = float(log_loss(y_va, np.clip(p_va, EPS_PROBA, 1 - EPS_PROBA), labels=[0, 1]))
    return (
        auc if np.isfinite(auc) else np.nan,
        boy if np.isfinite(boy) else np.nan,
        ll  if np.isfinite(ll)  else np.nan,
    )

## 5. Ensemble Weights & EvaluationShared by both models: AUC-based fold weighting and weighted-average ensemble on test data.

In [ ]:
def fold_weights_from_auc(cv_df, auc_col="best_val_auc", eps=1e-6, shift_by_min=True):
    """
    Build normalised ensemble weights from per-fold AUC.
    shift_by_min: w_k ∝ max(AUC_k − min(AUC), 0) + eps
    """
    cv_df = cv_df.sort_values("fold").reset_index(drop=True)
    folds = cv_df["fold"].to_numpy(dtype=int)
    aucs  = cv_df[auc_col].to_numpy(dtype=float)

    good = np.isfinite(aucs)
    if not good.any() or aucs.size == 0:
        w = np.ones_like(aucs)
    elif shift_by_min:
        amin = float(np.min(aucs[good]))
        w = np.where(good, np.maximum(aucs - amin, 0.0), 0.0)
        if w.max() <= 0:
            w = np.ones_like(aucs)
        w += eps
    else:
        w = np.where(good, aucs, 0.0) + eps

    return folds, (w / w.sum()).astype(np.float64)


def evaluate_ensemble_on_test(
    models, fold_weights, feature_cols, predict_fn,
    test_path=TEST_PATH, label_col=LABEL_COL,
):
    """
    Weighted-average ensemble evaluation on held-out test.

    predict_fn(model, X_clean) -> 1-D array of scores
    """
    df = pd.read_parquet(test_path).copy()
    ensure_binary_label(df, col=label_col)

    avail = [c for c in feature_cols if c in df.columns]
    if not avail:
        raise RuntimeError("No training features found in test parquet.")
    missing = [c for c in feature_cols if c not in df.columns]
    if missing:
        warnings.warn(f"Test missing {len(missing)} feature(s): {missing}")

    X = _nan_to_zero(df[avail].astype(float).to_numpy())
    y = df[label_col].to_numpy(dtype=int)

    K = len(models)
    w = np.asarray(fold_weights, dtype=np.float64)
    if w.shape[0] != K or not np.isfinite(w).all() or w.sum() <= 0:
        w = np.ones(K) / K
    else:
        w = w / w.sum()

    P = np.stack([predict_fn(m, X) for m in models])
    p_ens = (w[:, None] * P).sum(axis=0)

    m = compute_metrics(y, p_ens, nbins_boyce=NBINS_BOYCE)
    out = dict(test_Boyce=float(m["Boyce"]), test_AUC=float(m["ROC_AUC"]),
               n_test=len(y), n_pos=int(y.sum()), n_bg=int((1 - y).sum()))

    print("\n===== HELD-OUT TEST (ENSEMBLE) =====")
    for k, v in out.items():
        print(f"  {k}: {v}")

    return out, p_ens, y, avail

## 6. Tiled Map Prediction (shared)

In [ ]:
def build_suitability_map_ensemble(
    raster_dir, landmask_tif, out_dir, models, fold_weights,
    feature_cols, predict_fn, output_name,
    tile=TILE, dst_nodata=DST_NODATA, landmask_value=LANDMASK_VALUE,
):
    """
    Tiled suitability map from an ensemble of fold models.
    Rasters must already be standardised.

    predict_fn(model, X_df_or_array) -> 1-D array of scores
    """
    os.makedirs(out_dir, exist_ok=True)
    K = len(models)
    w = np.asarray(fold_weights, dtype=np.float64)
    if w.shape[0] != K or not np.isfinite(w).all() or w.sum() <= 0:
        w = np.ones(K) / K
    else:
        w = w / w.sum()

    ras_paths = sorted(Path(raster_dir).glob("*.tif"))
    if not ras_paths:
        raise FileNotFoundError(f"No rasters in {raster_dir}")
    name_to_path = {p.stem: p for p in ras_paths}

    present = [f for f in feature_cols if f in name_to_path]
    missing = [f for f in feature_cols if f not in name_to_path]
    if missing:
        warnings.warn(f"Missing {len(missing)} raster(s), filling with 0: {missing}")
    if not present:
        raise RuntimeError("None of feature_cols found as rasters.")

    with rasterio.open(str(name_to_path[present[0]])) as ref:
        ref_sig, profile = _grid_signature(ref), ref.profile.copy()
        H, W = ref.height, ref.width

    for f in present:
        with rasterio.open(str(name_to_path[f])) as src:
            if not _same_grid(_grid_signature(src), ref_sig):
                raise RuntimeError(f"Grid mismatch: {name_to_path[f].name}")

    with rasterio.open(landmask_tif) as lm:
        if not _same_grid(_grid_signature(lm), ref_sig):
            raise RuntimeError("Landmask grid mismatch.")
        landmask = lm.read(1)

    out_map = np.full((H, W), np.nan, dtype=np.float32)
    n_tiles = ((H + tile - 1) // tile) * ((W + tile - 1) // tile)

    for r0, r1, c0, c1 in tqdm(tiles(H, W, tile), total=n_tiles, desc="Map"):
        lm_tile = landmask[r0:r1, c0:c1]
        if landmask_value is not None and not np.any(lm_tile == landmask_value):
            continue
        rc = np.argwhere(lm_tile == landmask_value) if landmask_value is not None \
            else np.argwhere(np.isfinite(lm_tile))
        if rc.size == 0:
            continue
        rr, cc = rc[:, 0], rc[:, 1]
        n = rr.size

        X = np.empty((n, len(feature_cols)), dtype=np.float32)
        win = Window(c0, r0, c1 - c0, r1 - r0)
        for j, fname in enumerate(feature_cols):
            if fname not in name_to_path:
                X[:, j] = 0.0; continue
            with rasterio.open(str(name_to_path[fname])) as src:
                a = np.array(src.read(1, window=win, masked=True).filled(np.nan), dtype="float32")
            X[:, j] = np.nan_to_num(a[rr, cc], nan=0.0, posinf=0.0, neginf=0.0)

        # weighted ensemble
        p_ens = sum(w[k] * predict_fn(models[k], X) for k in range(K))
        out_map[r0 + rr, c0 + cc] = p_ens.astype(np.float32)

    prof = profile.copy()
    prof.update(count=1, dtype="float32", nodata=dst_nodata)
    out_path = os.path.join(out_dir, output_name)
    with rasterio.open(out_path, "w", **prof) as dst:
        dst.write(np.where(np.isfinite(out_map), out_map, dst_nodata).astype("float32"), 1)
    print(f"Written: {out_path}")
    return out_path

## 7. Generic CV DriverA single cross-validation function used by both models. The caller provides:- `fit_fn(X_train, y_train, stage_value) → model`- `predict_fn(model, X) → scores`- `stage_values` — list of hyperparameter values to search- `stage_name` — label for the hyperparameter (e.g. "trees" or "beta")

In [ ]:
def run_cv_baseline(
    parquet_path, fit_fn, predict_fn, stage_values, stage_name,
    fold_col=FOLD_COL, label_col=LABEL_COL,
):
    """
    K-fold CV with stage search and OOF metrics.

    fit_fn(X_train_df, y_train, stage_value) -> model
    predict_fn(model, X_clean) -> 1-D array of scores
    """
    df = pd.read_parquet(parquet_path).copy()
    df = df.dropna(subset=[fold_col, label_col]).reset_index(drop=True)
    ensure_binary_label(df, col=label_col)

    feature_cols = select_numeric_features(df)
    if not feature_cols:
        raise RuntimeError("No numeric feature columns found.")

    folds = np.sort(df[fold_col].dropna().unique().astype(int))
    models_per_fold = {}
    oof_pred = np.full(len(df), np.nan, dtype=np.float64)
    per_fold_rows, stage_rows = [], []
    y_all = df[label_col].to_numpy(dtype=int)

    for k in folds:
        k_int = int(k)
        tr = (df[fold_col] != k).to_numpy()
        va = (df[fold_col] == k).to_numpy()

        X_tr = _nan_to_zero(df.loc[tr, feature_cols].astype(float))
        y_tr = df.loc[tr, label_col].to_numpy(dtype=int)
        X_va = _nan_to_zero(df.loc[va, feature_cols].astype(float))
        y_va = df.loc[va, label_col].to_numpy(dtype=int)

        aucs, boys, lls = [], [], []
        stage_models = []

        for sv in stage_values:
            mdl = fit_fn(X_tr, y_tr, sv)
            p_va = predict_fn(mdl, X_va if isinstance(X_va, np.ndarray) else X_va.to_numpy())
            auc, boy, ll = _eval_stage(y_va, p_va)
            aucs.append(auc); boys.append(boy); lls.append(ll)
            stage_models.append(mdl)

            stage_rows.append({"fold": k_int, stage_name: sv,
                               "val_auc": auc, "val_boyce": boy, "val_logloss": ll})

        best_idx = select_best_stage(np.array(aucs), np.array(boys), np.array(lls))
        best_sv = stage_values[best_idx]

        # refit on same train split at best stage
        mdl_best = fit_fn(X_tr, y_tr, best_sv)
        models_per_fold[k_int] = mdl_best

        p_oof = predict_fn(mdl_best, X_va if isinstance(X_va, np.ndarray) else X_va.to_numpy())
        oof_pred[np.where(va)[0]] = p_oof

        row = {"fold": k_int, f"best_{stage_name}": best_sv,
               "best_val_auc": aucs[best_idx], "best_val_boyce": boys[best_idx],
               "best_val_logloss": lls[best_idx]}
        per_fold_rows.append(row)

        print(f"[fold {k_int}] {stage_name}={best_sv} | "
              f"AUC={aucs[best_idx]:.4f}  Boyce={boys[best_idx]:.4f}  "
              f"logloss={lls[best_idx]:.4f}")

    m_oof = compute_metrics(y_all, oof_pred, nbins_boyce=NBINS_BOYCE)
    oof_metrics = {"OOF_Boyce": float(m_oof["Boyce"]), "OOF_AUC": float(m_oof["ROC_AUC"])}

    cv_df = pd.DataFrame(per_fold_rows).sort_values("fold").reset_index(drop=True)
    stage_df = pd.DataFrame(stage_rows)

    print(f"\nOOF metrics: {oof_metrics}")
    print(cv_df)

    return dict(cv_df=cv_df, stage_df=stage_df, feature_cols=feature_cols,
                models_per_fold=models_per_fold, oof_pred=oof_pred, oof_metrics=oof_metrics)

## 8. Load Data

In [ ]:
df_train = pd.read_parquet(TRAIN_PATH)
print(f"Train: {df_train.shape}")
print(f"Test:  {pd.read_parquet(TEST_PATH).shape}")

---## Model 1 — Random ForestSuitability = P(y=1|x) from `predict_proba`. Stage search over `n_estimators`.

In [ ]:
def _rf_fit(X_tr, y_tr, n_trees):
    params = dict(RF_PARAMS, n_estimators=int(n_trees))
    if CLASS_WEIGHT_BALANCED:
        params["class_weight"] = "balanced"
    return RandomForestClassifier(**params).fit(X_tr, y_tr)

def _rf_predict(model, X):
    return model.predict_proba(X)[:, 1].astype(np.float64)

rf_out = run_cv_baseline(
    TRAIN_PATH, _rf_fit, _rf_predict, list(TREE_STAGES), "trees",
)

rf_folds, rf_w = fold_weights_from_auc(rf_out["cv_df"], shift_by_min=AUC_WEIGHT_SHIFT_BY_MIN)
rf_models = [rf_out["models_per_fold"][int(f)] for f in rf_folds]

rf_test, rf_scores, rf_y, _ = evaluate_ensemble_on_test(
    rf_models, rf_w, rf_out["feature_cols"], _rf_predict,
)

build_suitability_map_ensemble(
    PROCESSED_RASTER_DIR, LANDMASK_TIF, OUT_DIR_RF,
    rf_models, rf_w, rf_out["feature_cols"], _rf_predict,
    "rf_suitability_mean_weighted.tif",
)

---## Model 2 — MaxEnt (elapid)Suitability from `model.predict(X)` with cloglog transform. Stage search over `beta_multiplier`.

In [ ]:
def _maxent_fit(X_tr, y_tr, beta):
    mdl = ela.MaxentModel(transform=MAXENT_TRANSFORM, beta_multiplier=float(beta))
    if isinstance(X_tr, np.ndarray):
        X_tr = pd.DataFrame(X_tr)
    mdl.fit(X_tr, y_tr)
    return mdl

def _maxent_predict(model, X):
    try:
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X)
        return np.asarray(model.predict(X), dtype=np.float64)
    except Exception:
        return np.full(len(X), np.nan, dtype=np.float64)

me_out = run_cv_baseline(
    TRAIN_PATH, _maxent_fit, _maxent_predict, list(BETA_STAGES), "beta",
)

me_folds, me_w = fold_weights_from_auc(me_out["cv_df"], shift_by_min=AUC_WEIGHT_SHIFT_BY_MIN)
me_models = [me_out["models_per_fold"][int(f)] for f in me_folds]

me_test, me_scores, me_y, _ = evaluate_ensemble_on_test(
    me_models, me_w, me_out["feature_cols"], _maxent_predict,
)

build_suitability_map_ensemble(
    PROCESSED_RASTER_DIR, LANDMASK_TIF, OUT_DIR_MAXENT,
    me_models, me_w, me_out["feature_cols"], _maxent_predict,
    "maxent_suitability_mean_weighted.tif",
)

---## Summary

In [ ]:
print("=" * 55)
print(f"{'Model':<20} {'CV AUC':>10} {'CV Boyce':>10} {'Test AUC':>10} {'Test Boyce':>10}")
print("-" * 55)
for name, cv, t in [
    ("Random Forest", rf_out["oof_metrics"], rf_test),
    ("MaxEnt",        me_out["oof_metrics"], me_test),
]:
    print(f"{name:<20} {cv['OOF_AUC']:>10.4f} {cv['OOF_Boyce']:>10.4f} "
          f"{t['test_AUC']:>10.4f} {t['test_Boyce']:>10.4f}")
print("=" * 55)

## (Optional) Save Artefacts

In [ ]:
for prefix, out in [("rf", rf_out), ("maxent", me_out)]:
    out["cv_df"].to_csv(f"{prefix}_cv_summary.csv", index=False)
    out["stage_df"].to_csv(f"{prefix}_stage_history.csv", index=False)
    pd.DataFrame([out["oof_metrics"]]).to_csv(f"{prefix}_oof_metrics.csv", index=False)
    np.save(f"{prefix}_oof_pred.npy", out["oof_pred"])

for prefix, folds, w, scores in [
    ("rf", rf_folds, rf_w, rf_scores),
    ("maxent", me_folds, me_w, me_scores),
]:
    np.save(f"{prefix}_fold_ids.npy", folds)
    np.save(f"{prefix}_fold_weights.npy", w)
    np.save(f"{prefix}_test_scores.npy", scores)

print("All artefacts saved.")